# Data Error Sensitivity in Mean Reversion

This notebook shows why small quote errors can create large false signals in spread strategies.

It has two parts:

- a one-row arithmetic example matching Section 3.6;
- a local crypto spread with one injected bad tick.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "fixtures/crypto/crypto_daily_close.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Missing shared crypto fixture: fixtures/crypto/crypto_daily_close.csv. "
        "Run `python3 scripts/python/download-crypto-fixtures.py --source binance-monthly-archive` "
        "from the repository root."
    )


repo_root = find_repo_root()
close = pd.read_csv(
    repo_root / "fixtures/crypto/crypto_daily_close.csv",
    parse_dates=["date"],
).set_index("date").sort_index()
close = close.apply(pd.to_numeric, errors="coerce").dropna(how="all")

symbols = ["BTCUSDT", "ETHUSDT"]
missing = [symbol for symbol in symbols if symbol not in close]
if missing:
    raise ValueError(f"Fixture is missing required symbols: {missing}")

prices = close[symbols].dropna()
prices.tail()


In [ ]:
true_x = 100.0
true_y = 105.0
bad_y = 106.0

pd.Series(
    {
        "true_spread": true_y - true_x,
        "bad_spread": bad_y - true_x,
        "single_leg_price_error_pct": (bad_y - true_y) / true_y,
        "spread_error_pct": ((bad_y - true_x) - (true_y - true_x)) / (true_y - true_x),
    }
)


In [ ]:
lookback = 20
entry_threshold = 2
x = prices["BTCUSDT"]
y = prices["ETHUSDT"]

# Use a fixed hedge ratio from the first lookback window to make one bad tick easy to inspect.
design = np.column_stack([x.iloc[:lookback].to_numpy(), np.ones(lookback)])
hedge_ratio = np.linalg.lstsq(design, y.iloc[:lookback].to_numpy(), rcond=None)[0][0]
clean_spread = (y - hedge_ratio * x).rename("clean_spread")


def rolling_z(series: pd.Series, lookback: int) -> pd.Series:
    return (series - series.rolling(lookback).mean()) / series.rolling(lookback).std()


z_clean = rolling_z(clean_spread, lookback).rename("clean_zscore")

bad_tick_change_pct = 0.01
bad_tick_date = None
for candidate_date in y.index[lookback * 2 : -lookback]:
    candidate_y = y.copy()
    candidate_y.loc[candidate_date] *= 1 + bad_tick_change_pct
    candidate_z = rolling_z(candidate_y - hedge_ratio * x, lookback)
    changes_entry_set = ((candidate_z.abs() > entry_threshold) != (z_clean.abs() > entry_threshold)).any()
    if changes_entry_set:
        bad_tick_date = candidate_date
        break

if bad_tick_date is None:
    raise RuntimeError("Could not find a one-tick example that changes the entry set.")

dirty_y = y.copy()
dirty_y.loc[bad_tick_date] *= 1 + bad_tick_change_pct
dirty_spread = (dirty_y - hedge_ratio * x).rename("dirty_spread")

comparison = pd.concat([clean_spread, dirty_spread], axis=1)
comparison.loc[bad_tick_date - pd.Timedelta(days=3): bad_tick_date + pd.Timedelta(days=3)]

In [ ]:
z_dirty = rolling_z(dirty_spread, lookback).rename("dirty_zscore")

z_compare = pd.concat([z_clean, z_dirty], axis=1)
z_compare["zscore_difference"] = z_compare["dirty_zscore"] - z_compare["clean_zscore"]
z_compare.loc[bad_tick_date - pd.Timedelta(days=5): bad_tick_date + pd.Timedelta(days=5)]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

comparison.plot(ax=axes[0], linewidth=2)
axes[0].axvline(bad_tick_date, color="tab:red", linestyle="--", linewidth=1)
axes[0].set_title("One bad tick changes the spread")

z_compare[["clean_zscore", "dirty_zscore"]].plot(ax=axes[1], linewidth=2)
axes[1].axhline(2, color="tab:red", linestyle="--", linewidth=1)
axes[1].axhline(-2, color="tab:red", linestyle="--", linewidth=1)
axes[1].axvline(bad_tick_date, color="tab:red", linestyle="--", linewidth=1)
axes[1].set_title("The same bad tick can create or erase entry signals")

plt.tight_layout();


In [ ]:
entry_threshold = 2
clean_entries = z_clean.abs() > entry_threshold
dirty_entries = z_dirty.abs() > entry_threshold
false_entries = dirty_entries & ~clean_entries
missed_entries = clean_entries & ~dirty_entries

pd.Series(
    {
        "hedge_ratio": hedge_ratio,
        "bad_tick_date": str(bad_tick_date.date()),
        "bad_tick_price_change_pct": bad_tick_change_pct,
        "clean_entry_days": int(clean_entries.sum()),
        "dirty_entry_days": int(dirty_entries.sum()),
        "false_entry_days_from_dirty_data": int(false_entries.sum()),
        "missed_entry_days_from_dirty_data": int(missed_entries.sum()),
        "largest_abs_zscore_difference": float(z_compare["zscore_difference"].abs().max()),
    }
)


## Interpretation

A small error in one leg can be a large error in the spread because the spread is a small difference between larger prices. Mean-reversion rules are especially exposed because bad ticks often look like ideal temporary deviations followed by reversal.
